# Phase 2 — Ablation Study: Original VAE + LPIPS + Bilateral Post-processing

**Workflow:**
1. Clone repo + copy data from Drive
2. Install dependencies
3. **[5 min]** Tune `lambda_p` on 5 frames
4. **[~15 min]** Generate qualitative report figures (4 frames)
5. **[~3 h]** Full ablation sweep — Option 1 + Option 2 on 30 frames
6. Generate comparison tables and Pareto figure
7. Save all results to Drive

**Key change vs previous run:** VAE fine-tuned weights are **disabled**.  
We use the original `stabilityai/sd-vae-ft-mse` with LPIPS perceptual loss.

**Estimated total:** ~3.5 h, ~4 Colab Pro units

## Cell 1 — Clone repo

In [ ]:
import os

REPO_DIR = '/content/pfe-latent-attack'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/03aziz03/pfe-latent-attack.git {REPO_DIR}
else:
    print('Repo already cloned — pulling latest changes ...')
    !git -C {REPO_DIR} pull origin main

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

# Verify new scripts exist
for script in ['scripts/tune_lambda_p.py',
               'scripts/generate_qualitative.py',
               'scripts/run_ablation_sweep.py',
               'scripts/ablation_table.py',
               'configs/phase2_option1.yaml']:
    status = '✓' if os.path.exists(script) else '✗ MISSING — run git pull'
    print(f'  {status}  {script}')

## Cell 2 — Mount Drive and copy data

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
ASSETS = '/content/drive/MyDrive/pfe_assets'

# YOLOv8 weights
os.makedirs('runs/yolov8n_detrac', exist_ok=True)
if not os.path.exists('runs/yolov8n_detrac/best.pt'):
    shutil.copy(f'{ASSETS}/best.pt', 'runs/yolov8n_detrac/best.pt')
print(f'✓ best.pt ({os.path.getsize("runs/yolov8n_detrac/best.pt")//1024} KB)')

# Attack eval frames
if not os.path.exists('data/images_50'):
    shutil.copytree(f'{ASSETS}/images_50', 'data/images_50')
n50 = len([f for f in os.listdir('data/images_50') if f.endswith('.jpg')])
print(f'✓ data/images_50: {n50} frames')

# NOTE: finetune_seqs NOT needed for Option 1 (original VAE, no fine-tuning)
# Fine-tuned weights also NOT needed

# Create output directories
for d in ['results/ablation', 'results/qualitative/frames',
          'results/qualitative/panels', 'results/figures/png',
          'results/figures/pdf', 'results/iso_budget']:
    os.makedirs(d, exist_ok=True)
print('✓ Output directories ready')

## Cell 3 — Install dependencies

In [ ]:
!pip install lpips>=0.1.4 -q
!pip install -r requirements.txt -q

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
      if torch.cuda.is_available() else 'N/A')

## Cell 4 — Tune lambda_p (~5 min)

Runs the attack on **5 frames** across 6 values of `lambda_p`.  
Helps pick the best trade-off between DFR (attack effectiveness) and LPIPS (visual quality).  
Look for the **largest `lambda_p` where mean_DFR stays close to the unconstrained baseline** (lambda_p=0).

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python scripts/tune_lambda_p.py \
    --config configs/phase2_option1.yaml \
    --data   data/images_50 \
    --n_frames 5 \
    --eps_z  0.50

### Update lambda_p if needed

The script prints a recommendation.  
If the recommended value differs from `0.001`, update the config before continuing:

In [ ]:
# Run only if you need to change lambda_p based on the tuning output above
# Replace 0.001 with the recommended value

import re

LAMBDA_P = 0.001   # <-- CHANGE THIS if tuning recommends a different value

config_path = 'configs/phase2_option1.yaml'
text = open(config_path).read()
text = re.sub(r'(lambda_p:\s*)[\d.e+-]+', f'\\g<1>{LAMBDA_P}', text)
open(config_path, 'w').write(text)
print(f'lambda_p set to {LAMBDA_P} in {config_path}')

# Verify
for line in open(config_path):
    if 'lambda_p' in line:
        print(' ', line.rstrip())

## Cell 5 — Qualitative figures for the report (~15 min)

Runs the attack on **4 representative frames** and generates:  
- `f15_comparison_{stem}.png` — Clean | Option 1 | Option 2 | Heatmap (4-col panel)  
- `f16_filter_effect_{stem}.png` — Bilateral filter before/after comparison  
- `f17_ablation_grid.png` — Multi-frame ablation grid for the report  
- `f18_perturbation_heatmap.png` — Shared-vmax heatmap comparison  
- Individual PNGs: clean, opt1, opt2, mask, diff images  

**This step is fast — run it before the long sweep.**

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python scripts/generate_qualitative.py \
    --config  configs/phase2_option1.yaml \
    --data    data/images_50 \
    --frames  img00001 img00008 img00020 img00026 \
    --output  results/qualitative \
    --eps_z   0.50

In [ ]:
# Preview the report figures inline
from IPython.display import Image, display
from pathlib import Path

panels = sorted(Path('results/qualitative/panels').glob('*.png'))
print(f'Generated {len(panels)} panel figures:')
for p in panels:
    print(f'  {p.name}')
    display(Image(str(p), width=900))

## Cell 6 — Full ablation sweep (~3 h)

Runs Option 1 (original VAE + LPIPS) **and** Option 2 (+ bilateral filter) in one pass.  
Sweeps `eps_z ∈ {0.25, 0.50, 1.00}` + PGD baselines on **30 frames**.  
Use `--resume` to restart after a Colab disconnect without losing progress.

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python scripts/run_ablation_sweep.py \
    --config  configs/phase2_option1.yaml \
    --data    data/images_50 \
    --n_frames 30 \
    --output  results/ablation \
    --resume

In [ ]:
# Quick sanity check — print summary table
import json
summary = json.load(open('results/ablation/summary.json'))
print(f"{'Config':<35} {'DFR':>8} {'LPIPS':>8}")
print('-' * 55)
for tag, s in summary.items():
    dfr  = f"{s['mean_dfr']:+.4f}"   if s.get('mean_dfr')   else 'N/A'
    lpip = f"{s['mean_lpips']:.4f}"  if s.get('mean_lpips') else 'N/A'
    print(f"{tag:<35} {dfr:>8} {lpip:>8}")

## Cell 7 — Generate comparison tables and figures (~2 min)

Produces:  
- `ablation_dfr_bar.png` — DFR bar chart: Phase 1 vs Opt1 vs Opt2 vs PGD  
- `ablation_lpips_box.png` — LPIPS distribution box plots  
- `ablation_pareto.png` — DFR vs LPIPS Pareto scatter  
- `ablation_table.md` — Markdown table for the report  
- `ablation_table.csv` — CSV for further analysis

In [ ]:
!python scripts/ablation_table.py \
    --phase1_dir   results/iso_budget \
    --ablation_dir results/ablation \
    --output_dir   results/figures

In [ ]:
# Preview statistical figures
from IPython.display import Image, display
from pathlib import Path

for fig in ['ablation_dfr_bar.png', 'ablation_lpips_box.png', 'ablation_pareto.png']:
    p = Path('results/figures') / fig
    if p.exists():
        print(f'--- {fig} ---')
        display(Image(str(p), width=800))

## Cell 8 — Generate Pareto curve f14 (~1 min)

In [ ]:
# Use ablation summary (Option 1 + Option 2) as the Pareto source
!python scripts/generate_pareto.py \
    --summary results/ablation/summary.json

from IPython.display import Image
Image('results/figures/png/f14_pareto_dfr_lpips.png', width=700)

## Cell 9 — Print markdown table for the report

In [ ]:
table_path = 'results/figures/ablation_table.md'
if __import__('os').path.exists(table_path):
    print(open(table_path).read())
else:
    print('Table not found — run Cell 7 first.')

## Cell 10 — Save all results to Drive

In [ ]:
import shutil, os
from pathlib import Path

ASSETS = '/content/drive/MyDrive/pfe_assets'

# Ablation sweep results
shutil.copytree('results/ablation',
                f'{ASSETS}/ablation',
                dirs_exist_ok=True)
print('✓ results/ablation → Drive')

# Qualitative figures
shutil.copytree('results/qualitative',
                f'{ASSETS}/qualitative',
                dirs_exist_ok=True)
print('✓ results/qualitative → Drive')

# Statistical figures + tables
shutil.copytree('results/figures',
                f'{ASSETS}/figures',
                dirs_exist_ok=True)
print('✓ results/figures → Drive')

# List everything saved
print('\nFiles saved to Drive:')
for root, dirs, files in os.walk(ASSETS):
    dirs[:] = [d for d in dirs if d not in ['__pycache__']]
    level = root.replace(ASSETS, '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    if level <= 2:
        print(f'{indent}{folder}/')
    for f in files:
        if level <= 2:
            print(f'{indent}  {f}')